In [1]:
%reload_ext autoreload
%autoreload 2
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

In [2]:
from transformers import ViTModel, ViTConfig
import torch
from torch import nn
from torch.optim import optimizer
from torch.optim import Adam

In [3]:
import configparser, os

config = configparser.ConfigParser()
config.read('config.ini')
# Load parameters from config file
#root = '/Users/leonjye/Documents/MachineLearingData'
root = config.get('DEFAULT', 'root_dir')
train_data_dir = os.path.join(root, 'CatAndDog', 'training_set')
val_data_dir = os.path.join(root, 'CatAndDog', 'test_set')

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    #使用ImageNet的均值和标准差进行归一化
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_set = datasets.ImageFolder(root=train_data_dir, transform=transform)
val_set = datasets.ImageFolder(root=val_data_dir, transform=transform)

# 取得每一類的 index
cat_indices = [i for i, (_, label) in enumerate(train_set.samples) if label == 0][:500]
dog_indices = [i for i, (_, label) in enumerate(train_set.samples) if label == 1][:500]
selected_indices = cat_indices + dog_indices

train_subset = Subset(train_set, selected_indices)

batch_size = 64
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)


In [5]:
next(iter(train_loader))  # Get a batch of data to check the shape

[tensor([[[[-1.2274, -1.1418, -1.0562,  ..., -0.5253, -0.5596, -0.5767],
           [-1.2788, -1.1760, -1.0562,  ..., -0.2684, -0.2856, -0.3027],
           [-1.3644, -1.2274, -1.0733,  ...,  0.2282,  0.2453,  0.2453],
           ...,
           [-0.8335, -0.8335, -0.7137,  ..., -0.3198, -0.2856, -0.3541],
           [-0.5424, -0.6281, -0.5596,  ..., -0.2342, -0.2856, -0.4054],
           [-0.4054, -0.5253, -0.4911,  ..., -0.1999, -0.2856, -0.4397]],
 
          [[-1.2304, -1.1429, -1.0553,  ..., -0.3200, -0.3550, -0.3725],
           [-1.2654, -1.1604, -1.0553,  ..., -0.0574, -0.0749, -0.0924],
           [-1.3529, -1.2129, -1.0553,  ...,  0.4503,  0.4678,  0.4678],
           ...,
           [-0.3550, -0.3550, -0.2325,  ...,  0.1176,  0.1527,  0.0826],
           [-0.0574, -0.1450, -0.0749,  ...,  0.2052,  0.1527,  0.0301],
           [ 0.0826, -0.0399, -0.0049,  ...,  0.2402,  0.1527, -0.0049]],
 
          [[-1.4559, -1.3687, -1.2816,  ..., -0.0267, -0.0615, -0.0790],
           [-

In [6]:
ViTConfig()

ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 224,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "pooler_act": "tanh",
  "pooler_output_size": 768,
  "qkv_bias": true,
  "transformers_version": "4.52.4"
}

In [7]:
model_checkpoint = 'google/vit-base-patch16-224-in21k'
class ViT(nn.Module):
  def __init__(self, config=ViTConfig(), num_labels=1,
               model_checkpoint=model_checkpoint):
        super(ViT, self).__init__()
        self.vit = ViTModel.from_pretrained(model_checkpoint, add_pooling_layer=False)
        self.classifier1 = (
            nn.Linear(config.hidden_size, 128)
        )
        self.classifier2 = (
            nn.Linear(128, num_labels)
        )
        self.classifier = nn.Sequential(
            self.classifier1,
            nn.ReLU(),
            self.classifier2)
        for param in self.vit.parameters():
            param.requires_grad = False

  def forward(self, x):
    x = self.vit(x)['last_hidden_state']
    # Use the embedding of [CLS] token
    output = self.classifier(x[:, 0, :])
    output = torch.sigmoid(output)
    return output

In [8]:
import numpy as np
class Report:
    def __init__(self, n_epochs):
        self.n_epochs = n_epochs
        self.epoch = 0
        self.trn_loss = []
        self.trn_acc = []
        self.val_acc = []

    def record(self, epoch, trn_loss=None, trn_acc=None, val_acc=None, end='\n'):
        if trn_loss is not None:
            self.trn_loss.append(trn_loss)
        if trn_acc is not None:
            self.trn_acc.append(trn_acc)
        if val_acc is not None:
            self.val_acc.append(val_acc)
        print(f'Epoch {epoch}/{self.n_epochs} - '
              f'Train Loss: {np.mean(self.trn_loss):.4f}, '
              f'Train Acc: {np.mean(self.trn_acc):.4f}, '
              f'Val Acc: {np.mean(self.val_acc):.4f}', end=end)

    def report_avgs(self, epoch):
        print(f'\nEpoch {epoch} - '
              f'Avg Train Loss: {np.mean(self.trn_loss):.4f}, '
              f'Avg Train Acc: {np.mean(self.trn_acc):.4f}, '
              f'Avg Val Acc: {np.mean(self.val_acc):.4f}')
        self.trn_loss.clear()
        self.trn_acc.clear()
        self.val_acc.clear()
@torch.no_grad()
def accuracy(x, y, model):
    model.eval()
    prediction = model(x)
    is_correct = (prediction > 0.5) == y
    return is_correct.cpu().numpy().tolist()

In [9]:
#model = ViT().to('cuda')
model = ViT()
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-3)

n_epochs = 3
#report = Report(n_epochs)
for epoch in range(n_epochs):
    train_epoch_losses, train_epoch_accuracies = [], []
    val_epoch_accuracies = []
    n = len(train_loader)
    for ix, batch in enumerate(iter(train_loader)):
        x, y = batch
        y = y.unsqueeze(1).float()  # Ensure y is of shape (batch_size, 1)
        model.train()
        prediction = model(x)
        batch_loss = loss_fn(prediction, y)
        batch_loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        is_correct = accuracy(x, y, model)
        #report.record(epoch+(ix+1)/n, trn_loss=batch_loss, trn_acc=np.mean(is_correct), end='\r')

    n = len(val_loader)
    for ix, batch in enumerate(iter(val_loader)):
        x, y = batch
        val_is_correct = accuracy(x, y, model)
        #report.record(epoch+(ix+1)/n, val_acc=np.mean(val_is_correct), end='\r')
    print(f'Epoch {epoch+1}/{n_epochs}, '
          f'Train Loss: {batch_loss.item():.4f}, '
          f'Train Accuracy: {np.mean(is_correct):.4f}, '
          f'Validation Accuracy: {np.mean(val_is_correct):.4f}')
    #report.report_avgs(epoch+1)

Some weights of the model checkpoint at google/vit-base-patch16-224-in21k were not used when initializing ViTModel: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Epoch 1/3, Train Loss: 0.1261, Train Accuracy: 1.0000, Validation Accuracy: 1.0000
Epoch 2/3, Train Loss: 0.0907, Train Accuracy: 0.9750, Validation Accuracy: 1.0000
Epoch 3/3, Train Loss: 0.0071, Train Accuracy: 1.0000, Validation Accuracy: 1.0000
